# Netflix Content Analysis — Data Cleaning

Dataset: Netflix Movies and TV Shows (Kaggle)  
Source: https://www.kaggle.com/datasets/shivamb/netflix-shows

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/raw/netflix_titles.csv')
print(f'{df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# check nulls
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'count': missing, '%': pct})[missing > 0]

In [ ]:
df['director'] = df['director'].fillna('Unknown')
df['cast']     = df['cast'].fillna('Unknown')
df['country']  = df['country'].fillna(df['country'].mode()[0])
df['rating']   = df['rating'].fillna(df['rating'].mode()[0])
df.dropna(subset=['date_added'], inplace=True)

print('nulls remaining:', df.isnull().sum().sum())

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['month_name']  = df['date_added'].dt.strftime('%b')

df[['date_added', 'year_added', 'month_added', 'month_name']].head()

In [ ]:
# duration column mixes '90 min' and '2 Seasons' — split into two numeric columns
df['duration_value']   = df['duration'].str.extract('(\d+)').astype(float)
df['duration_mins']    = df['duration_value'].where(df['type'] == 'Movie')
df['duration_seasons'] = df['duration_value'].where(df['type'] == 'TV Show')

df[['type', 'duration', 'duration_mins', 'duration_seasons']].head(8)

In [ ]:
# genres and countries have comma-separated values — explode so each row = one value
df_genres = df.copy()
df_genres['listed_in'] = df_genres['listed_in'].str.split(', ')
df_genres = df_genres.explode('listed_in').assign(listed_in=lambda x: x['listed_in'].str.strip())

df_country = df.copy()
df_country['country'] = df_country['country'].str.split(', ')
df_country = df_country.explode('country').assign(country=lambda x: x['country'].str.strip())

print(f'genres df:  {len(df_genres):,} rows, {df_genres["listed_in"].nunique()} unique genres')
print(f'country df: {len(df_country):,} rows, {df_country["country"].nunique()} unique countries')

In [ ]:
df.to_csv('../data/processed/netflix_clean.csv', index=False)
df_genres.to_csv('../data/processed/netflix_genres.csv', index=False)
df_country.to_csv('../data/processed/netflix_country.csv', index=False)

print(f'netflix_clean.csv   {len(df):,} rows')
print(f'netflix_genres.csv  {len(df_genres):,} rows')
print(f'netflix_country.csv {len(df_country):,} rows')